In [15]:
# 준비
import random

import torch
from torch import nn

In [16]:
# 데이터 생성 및 데이터 선택
# 긍정 문구
GOOD = ["음식이 맛있었다", "직원이 친절했다", "배송이 빨랐다", "양이 넉넉했다",
        "매장이 깨끗했다", "포장이 꼼꼼했다", "가격이 저렴했다", "분위기가 좋았다"]
# 부정 문구
BAD = ["음식이 짰다", "직원이 불친절했다", "배송이 늦었다", "양이 적었다",
       "매장이 지저분했다", "포장이 엉망이었다", "가격이 비쌌다", "분위기가 시끄러웠다"]
# 긍정문구 1개와 부정문구 1개를 선택해서 튜플로 만듬(총64개)
pairs = [(g, b) for g in GOOD for b in BAD]
# 섞음
random.Random(0).shuffle(pairs)

# 훈련(문장을 만들 재료), 테스트 비율(문장을 만들 재료) => 0.75:0.25
train_pairs, test_pairs = pairs[16:], pairs[:16]

# 선택된 부정 긍정을 이용하여 (문장과 정답) 튜플을 2개씩 만들어 리스트에 저장
def make_rows(pairs):
    rows = []
    for g, b in pairs:
        rows.append((f"{g} 하지만 {b}", 0))
        rows.append((f"{b} 하지만 {g}", 1))
    return rows

# 훈련데이터(문장), 테스트데이터(문장)
train_rows, test_rows = make_rows(train_pairs), make_rows(test_pairs)


In [17]:
# 사전준비 : 데이터 인코딩
# 단어 사전
# 문장에서 글자를 추출해서 글자에 위치를 부여
vocab = {w: i for i, w in enumerate(sorted({w for t, _ in train_rows + test_rows for w in t.split()}))}

# 단어 사전을 이용하여 문장들을 숫자로 변환
def encode(texts):
    return torch.tensor([[vocab[w] for w in t.split()] for t in texts])

# 훈련데이터와 테스트 데이터들을 인코딩(숫자로 변환)
X_train_t = encode([t for t, _ in train_rows])
X_test_t = encode([t for t, _ in test_rows])
y_train_t = torch.tensor([l for _, l in train_rows], dtype=torch.float32)
y_test_t = torch.tensor([l for _, l in test_rows])

In [18]:
# 모델 설계
class ReviewAttention(nn.Module):
    def __init__(self, use_position):
        super().__init__()
        self.use_position = use_position
        self.emb = nn.Embedding(len(vocab), 16)     # 단어 → 좌표
        self.pos = nn.Embedding(5, 16)              # 위치(0~4번째) → 좌표 (번호표)
        self.query = nn.Parameter(torch.randn(16))  # "결론을 내려면 무엇을 봐야 하나"를 학습할 질문
        self.out = nn.Linear(16, 1)

    def forward(self, x):
        e = self.emb(x)
        if self.use_position:
            e = e + self.pos(torch.arange(x.shape[1]))   # 단어 좌표에 위치 좌표를 더한다
        scores = e @ self.query                           # 단어마다 점수
        weights = torch.softmax(scores, dim=1)            # 비율로
        mixed = (weights.unsqueeze(-1) * e).sum(dim=1)    # 비율대로 섞기 (모든 단어를 한 번에)
        return self.out(mixed).squeeze(-1)

In [19]:
# 학습 
def train(model, epochs=300):
    loss_fn = nn.BCEWithLogitsLoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=0.01)
    for epoch in range(epochs):
        loss = loss_fn(model(X_train_t), y_train_t)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
    return model


def accuracy(model, X, y):
    with torch.no_grad():
        return ((model(X) > 0).long() == y.long()).float().mean().item()

In [20]:
torch.manual_seed(0)
# 번호표 없이 테스트
# 하지만을 이용해서 전후 사정을 제대로 이해하지 못함
no_pos = train(ReviewAttention(use_position=False))
print(f"학습 정확도: {accuracy(no_pos, X_train_t, y_train_t):.2f}")
print(f"시험 정확도: {accuracy(no_pos, X_test_t, y_test_t):.2f}")

학습 정확도: 0.50
시험 정확도: 0.50


In [21]:
# 번호표를 이용하여 테스트
with_pos = train(ReviewAttention(use_position=True))
print(f"학습 정확도: {accuracy(no_pos, X_train_t, y_train_t):.2f}")
print(f"시험 정확도: {accuracy(no_pos, X_test_t, y_test_t):.2f}")

학습 정확도: 0.50
시험 정확도: 0.50


In [35]:
pair = [
    "직원이 꼼꼼했다 하지만 가격이 비쌌다",
    "가격이 비쌌다 하지만 직원이 꼼꼼했다",
    "음식이 맛있었다 하지만 포장이 지저분했다",
    "음식이 맛있었다 하지만 포장이 지저분했다",
    # "직원이 꼼꼼했다 그리고 포장이 꼼꼼했다",
]
x = encode(pair)
with torch.no_grad():
    p_no = torch.sigmoid(no_pos(x))
    p_with = torch.sigmoid(with_pos(x))
for text, a, b in zip(pair, p_no, p_with):
    print(text)
    print(f"  번호표 없음: 긍정 확률 {a:.2f} / 번호표 있음: 긍정 확률 {b:.2f}")

직원이 꼼꼼했다 하지만 가격이 비쌌다
  번호표 없음: 긍정 확률 0.50 / 번호표 있음: 긍정 확률 0.00
가격이 비쌌다 하지만 직원이 꼼꼼했다
  번호표 없음: 긍정 확률 0.50 / 번호표 있음: 긍정 확률 1.00
음식이 맛있었다 하지만 포장이 지저분했다
  번호표 없음: 긍정 확률 0.50 / 번호표 있음: 긍정 확률 0.00
음식이 맛있었다 하지만 포장이 지저분했다
  번호표 없음: 긍정 확률 0.50 / 번호표 있음: 긍정 확률 0.00


In [ ]:
# 이 예제는 없는 단어거나 5 단어를 넘거나 부족하면 예측이 안되는 문제가 있음
#  => 인코딩 할 때 단어가 부족하면 0으로 채우는 과정이 없음
#  => 새로운 단어인 경우 새로운 단어임을 알리는 과정이 없음
# 이 조건 만족 한다면 정확하게 예측할 수 있다